# Module 5: OS Interaction & File Engineering
In this notebook, we are going to simulate a data-engineering pipeline. We will generate a messy folder of dummy dataset files, and then use Python to automatically analyze, sort, and organize them based on their metadata.

In [1]:
import os
import time

print("=== 1. Generating a Dummy Dataset ===")

# 1. Create a raw data folder using the OS module
raw_data_dir = "./raw_dataset"
os.makedirs(raw_data_dir, exist_ok=True)

# 2. Generate 5 dummy files with varying amounts of text (simulating different file sizes)
for i in range(1, 6):
    file_path = f"{raw_data_dir}/image_{i}.txt" # Simulating image labels
    with open(file_path, "w") as f:
        # We write more data to even numbered files to make them larger
        multiplier = 1000 if i % 2 == 0 else 10 
        f.write("DUMMY DATA " * multiplier)
        
    # Adding a tiny delay so they have slightly different creation timestamps
    time.sleep(0.1) 

print(f"[SUCCESS] Created 5 dummy files inside '{raw_data_dir}'.")

=== 1. Generating a Dummy Dataset ===
[SUCCESS] Created 5 dummy files inside './raw_dataset'.


## Reading Metadata & Permissions
Before we move files, a professional pipeline checks the metadata. We want to know how large these files are, and when they were last modified. We will use the `os.stat()` function for this.

In [2]:
import datetime

print("=== 2. Inspecting File Metadata ===")

# List all files in the directory
files = os.listdir(raw_data_dir)

for file_name in files:
    # We must combine the folder path with the file name to inspect it
    full_path = os.path.join(raw_data_dir, file_name)
    
    # Extract Metadata
    metadata = os.stat(full_path)
    
    # 1. Size in Bytes
    size_bytes = metadata.st_size
    
    # 2. Last Modified Time (Converted from a raw timestamp to a human-readable date)
    modified_timestamp = metadata.st_mtime
    readable_time = datetime.datetime.fromtimestamp(modified_timestamp).strftime('%Y-%m-%d %H:%M:%S')
    
    print(f"File: {file_name.ljust(15)} | Size: {size_bytes:5} bytes | Last Modified: {readable_time}")

=== 2. Inspecting File Metadata ===
File: image_1.txt     | Size:   110 bytes | Last Modified: 2026-04-22 12:28:32
File: image_2.txt     | Size: 11000 bytes | Last Modified: 2026-04-22 12:28:32
File: image_3.txt     | Size:   110 bytes | Last Modified: 2026-04-22 12:28:32
File: image_4.txt     | Size: 11000 bytes | Last Modified: 2026-04-22 12:28:32
File: image_5.txt     | Size:   110 bytes | Last Modified: 2026-04-22 12:28:32


## `pathlib` and `shutil`: Sorting the Data
Now, we will use modern `pathlib` to scan the directory safely across any Operating System. 
We will use `shutil` to move the files: Large files go into a `valid/` folder, and small files go into an `invalid/` folder.

In [3]:
import shutil
from pathlib import Path

print("=== 3. Organizing Files with Pathlib & Shutil ===")

# 1. Define our smart paths using Pathlib
base_path = Path("./raw_dataset")
valid_dir = Path("./processed/valid")
invalid_dir = Path("./processed/invalid")

# Create the destination folders (parents=True acts like os.makedirs)
valid_dir.mkdir(parents=True, exist_ok=True)
invalid_dir.mkdir(parents=True, exist_ok=True)

# 2. Iterate through all .txt files intelligently
# .glob() is a powerful pathlib feature that searches for specific file types
for file_path in base_path.glob("*.txt"):
    
    # We can get the size directly from the pathlib object!
    size = file_path.stat().st_size
    
    # 3. Sort using shutil
    if size > 5000:
        # Move to valid directory
        destination = valid_dir / file_path.name # Notice the magical '/' operator in pathlib!
        shutil.move(str(file_path), str(destination))
        print(f"Moved {file_path.name} -> VALID (Large enough)")
    else:
        # Move to invalid directory
        destination = invalid_dir / file_path.name
        shutil.move(str(file_path), str(destination))
        print(f"Moved {file_path.name} -> INVALID (Too small)")

print("\n[SUCCESS] Dataset completely sorted.")

=== 3. Organizing Files with Pathlib & Shutil ===
Moved image_1.txt -> INVALID (Too small)
Moved image_2.txt -> VALID (Large enough)
Moved image_3.txt -> INVALID (Too small)
Moved image_4.txt -> VALID (Large enough)
Moved image_5.txt -> INVALID (Too small)

[SUCCESS] Dataset completely sorted.


## Clean Up
In data engineering, you often need to clean up temporary folders after processing is done. `os.rmdir()` only works on empty folders. To delete a folder full of data, we must use `shutil.rmtree()`.

In [4]:
print("=== 4. System Cleanup ===")

# We delete the raw directory, which is now empty, and the processed directory
try:
    shutil.rmtree("./raw_dataset")
    shutil.rmtree("./processed")
    print("[SUCCESS] All practice directories and files deleted from the hard drive.")
except Exception as e:
    print(f"Error during cleanup: {e}")

=== 4. System Cleanup ===
[SUCCESS] All practice directories and files deleted from the hard drive.
